# Gestione delle partite terminate in modo errato

Questo notebook aiuta a identificare e cancellare dal database le partite *finished* che sono state terminate con `/finegioco` senza una vera partita giocata.

Le query si basano sullo schema attuale: se una partita è `finished` ma non ha mani (`hands`), molto probabilmente è stata chiusa in modo errato.

## 1. Connetti al database

Carica le variabili d'ambiente e stabilisci una connessione a Supabase. Assicurati di avere `SUPABASE_URL` e `SUPABASE_KEY` nel file `.env`.

In [ ]:
import os
from dotenv import load_dotenv
from supabase import acreate_client

load_dotenv()

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')

print('SUPABASE_URL:', 'ok' if SUPABASE_URL else 'manca')
print('SUPABASE_KEY:', 'ok' if SUPABASE_KEY else 'manca')

async def connect():
    if not SUPABASE_URL or not SUPABASE_KEY:
        raise ValueError('Devi avere SUPABASE_URL e SUPABASE_KEY configurate nel file .env')
    client = await acreate_client(SUPABASE_URL, SUPABASE_KEY)
    return client

client = await connect()
print('Connessione stabilita')


SUPABASE_URL: ok
SUPABASE_KEY: ok
Connessione stabilita


/Users/giovanni_buonfrate/Repository/burraco_bot/.venv/lib/python3.12/site-packages/httpcore/_exceptions.py:72: RuntimeWarning: coroutine 'connect' was never awaited
  class ConnectError(NetworkError):


## 2. Query delle partite errate

Identifichiamo le partite finite (`status = 'finished'`) senza mani registrate nella tabella `hands`. Queste sono le principali candidate per la cancellazione.

In [11]:
import pandas as pd

print('Query candidate per partite finite senza mani:')

finished_res = await (
    client.table('games')
    .select('id, chat_id, chat_title, created_by, winner_id, target_score, created_at, finished_at')
    .eq('status', 'finished')
    .execute()
)

if finished_res is None or getattr(finished_res, 'data', None) is None:
    raise RuntimeError('Errore durante il recupero delle partite finished')

finished_games = pd.DataFrame(finished_res.data)
print('Partite finished trovate:', len(finished_games))

if not finished_games.empty:
    game_ids = finished_games['id'].tolist()
    hands_res = await (
        client.table('hands')
        .select('game_id')
        .in_('game_id', game_ids)
        .execute()
    )
    hands = pd.DataFrame(hands_res.data or [])
    hand_counts = hands['game_id'].value_counts().to_dict() if not hands.empty else {}
    finished_games['hand_count'] = finished_games['id'].map(hand_counts).fillna(0).astype(int)
    df_candidates = finished_games[finished_games['hand_count'] == 0].copy()
else:
    df_candidates = pd.DataFrame([])

display(df_candidates)


Query candidate per partite finite senza mani:
Partite finished trovate: 17


,id,chat_id,chat_title,created_by,winner_id,target_score,created_at,finished_at,hand_count


## 3. Visualizza i giochi prima della cancellazione

Mostra i dettagli dei giochi candidati, i giocatori coinvolti e i punteggi totali.

In [8]:
async def show_game_players(game_ids):
    if not game_ids:
        print('Nessun game_id fornito')
        return

    players_res = await (
        client.table('game_players')
        .select('game_id, player_id, total_score, players(display_name, username)')
        .in_('game_id', game_ids)
        .execute()
    )

    if players_res is None or getattr(players_res, 'data', None) is None:
        print('Impossibile caricare i giocatori delle partite. Usa un client SQL diretto se necessario.')
        return

    df_players = pd.DataFrame(players_res.data)
    display(df_players)

if 'df_candidates' in globals() and not df_candidates.empty:
    print('Giochi candidati alla cancellazione:')
    display(df_candidates)
    await show_game_players(df_candidates['id'].tolist())
else:
    print('Nessuna partita errata trovata con la query corrente.')


Giochi candidati alla cancellazione:


,id,chat_id,chat_title,created_by,winner_id,target_score,created_at,finished_at,hand_count
2,11,317406998,Gruppo,317406998,317406998,2000,2026-04-14T19:37:35.182637+00:00,2026-04-15T22:16:20.429575+00:00,0
14,5,-5258182851,Burros,317406998,317406998,2000,2026-03-24T22:32:29.677582+00:00,2026-03-25T09:19:42.978775+00:00,0
17,8,-5258182851,Burros,317406998,317406998,2000,2026-03-26T22:26:06.533351+00:00,2026-03-26T22:27:31.763667+00:00,0


,game_id,player_id,total_score,players
0,5,317406998,0,"{'username': 'Giov_7', 'display_name': 'Giovan..."
1,8,317406998,0,"{'username': 'Giov_7', 'display_name': 'Giovan..."
2,8,1112469803,0,"{'username': 'MarcoCarpinteri', 'display_name'..."
3,11,317406998,0,"{'username': 'Giov_7', 'display_name': 'Giovan..."


## 4. Cancella le partite errate

Rimuove le partite candidate e tutte le righe correlate (`game_players`, `hands`, `hand_scores`) grazie a `ON DELETE CASCADE`.

Usa questa cella solo dopo aver verificato le partite nel passaggio precedente.

In [9]:
if 'df_candidates' in globals() and not df_candidates.empty:
    ids_to_delete = df_candidates['id'].tolist()
    print('Cancello game_id:', ids_to_delete)

    delete_res = await (
        client.table('games')
        .delete()
        .in_('id', ids_to_delete)
        .execute()
    )

    if delete_res is None or getattr(delete_res, 'error', None) is not None:
        print('Errore durante la cancellazione:', getattr(delete_res, 'error', 'Unknown error'))
    else:
        print('Cancellazione completata.')
else:
    print('Nessuna partita selezionata per la cancellazione.')


Cancello game_id: [11, 5, 8]
Cancellazione completata.


## 5. Visualizza i giochi dopo la cancellazione

Verifica che i giochi candidati siano stati rimossi.

In [10]:
finished_res = await (
    client.table('games')
    .select('id, chat_id, chat_title, created_by, winner_id, target_score, created_at, finished_at')
    .eq('status', 'finished')
    .execute()
)

if finished_res is None or getattr(finished_res, 'data', None) is None:
    raise RuntimeError('Errore durante il recupero delle partite finished per verifica')

finished_games = pd.DataFrame(finished_res.data)

if not finished_games.empty:
    game_ids = finished_games['id'].tolist()
    hands_res = await (
        client.table('hands')
        .select('game_id')
        .in_('game_id', game_ids)
        .execute()
    )
    hands = pd.DataFrame(hands_res.data or [])
    hand_counts = hands['game_id'].value_counts().to_dict() if not hands.empty else {}
    finished_games['hand_count'] = finished_games['id'].map(hand_counts).fillna(0).astype(int)
    df_after = finished_games[finished_games['hand_count'] == 0].copy()
else:
    df_after = pd.DataFrame([])

if not df_after.empty:
    display(df_after)
print('Righe trovate:', len(df_after))


Righe trovate: 0
